# Stage 7 — Error Detector
Uses **sympy** to validate each step, then **Qwen3** (via Groq) to identify the specific error.

**Dataset structure:** `{equation, steps[]}` — no error_type labels.  
Errors are detected by validating each step mathematically.

**Pipeline per entry:**
1. Validate each step with sympy
2. Find the first wrong step
3. Call Qwen3 to describe the specific error
4. Output: `{equation, wrong_step, prev_step, error_type, error_detail, confidence}`

In [32]:
%pip install groq python-dotenv sympy --quiet

Note: you may need to restart the kernel to use updated packages.


In [33]:
import json
import re
import time
from groq import Groq
from dotenv import load_dotenv
from sympy import symbols, expand, simplify
from sympy.parsing.sympy_parser import parse_expr, standard_transformations

load_dotenv()
client = Groq()
x = symbols('x')

MODEL = "qwen/qwen3-32b"

## Step 1 — Sympy validator
Converts student notation to sympy and checks if the step is mathematically correct.

In [34]:
def to_sympy_notation(expr: str) -> str:
    """Convert student notation to sympy-parseable string."""
    expr = expr.strip()
    expr = re.sub(r'=\s*0', '', expr)                        # remove = 0
    expr = expr.replace('^', '**')                           # x^2 → x**2
    expr = re.sub(r'\)\s*\(', ')*(', expr)                   # )( → )*(
    expr = re.sub(r'(\d)\s*\(', r'\1*(', expr)               # 2( → 2*(
    expr = re.sub(r'(\d)\s*([a-zA-Z])', r'\1*\2', expr)      # 5x → 5*x
    return expr.strip()


def validate_factorization(equation: str, factored: str) -> bool | None:
    """Check expand(factored) == original equation LHS. Returns True/False/None."""
    try:
        orig = parse_expr(to_sympy_notation(equation),
                          transformations=standard_transformations,
                          local_dict={"x": x})
        fact = parse_expr(to_sympy_notation(factored),
                          transformations=standard_transformations,
                          local_dict={"x": x})
        return simplify(expand(fact) - orig) == 0
    except Exception:
        return None


def validate_arithmetic(expr: str) -> bool | None:
    """
    Check arithmetic/radical result steps.
    e.g. '36 - 0 = 40'  → False (correct answer is 36)
         '√12 = 12'     → False (√12 ≈ 3.46)
         '36 - 0 = 36'  → True
    """
    try:
        # Handle 'a OP b = c' patterns (discriminant steps)
        arith_match = re.match(r'^([\d\s\+\-\*\/\.]+)=([\d\s\.]+)$', expr.strip())
        if arith_match:
            lhs = eval(arith_match.group(1).strip())
            rhs = float(arith_match.group(2).strip())
            return abs(lhs - rhs) < 0.01

        # Handle '√N = M' pattern
        sqrt_match = re.match(r'^√(\d+)\s*=\s*([\d\.]+)$', expr.strip())
        if sqrt_match:
            import math
            n = float(sqrt_match.group(1))
            m = float(sqrt_match.group(2))
            return abs(math.sqrt(n) - m) < 0.01

        return None  # step type not recognised
    except Exception:
        return None


def find_first_error(entry: dict) -> dict | None:
    """
    Validate steps sequentially. Return info about the first wrong step.
    Returns None if no error detected.
    """
    steps = entry['steps']
    equation = entry['equation']

    for i in range(1, len(steps)):
        prev = steps[i - 1]
        curr = steps[i]

        # Check arithmetic/radical steps first
        arith_result = validate_arithmetic(curr)
        if arith_result is False:
            return {
                "equation":    equation,
                "error_index": i,
                "step_prev":   prev,
                "step_wrong":  curr,
                "detected_by": "sympy_arithmetic"
            }

        # Check factorization steps (contains parens, no OR)
        if '(' in curr and ')' in curr and 'OR' not in curr and '±' not in curr:
            fact_result = validate_factorization(equation, curr)
            if fact_result is False:
                return {
                    "equation":    equation,
                    "error_index": i,
                    "step_prev":   prev,
                    "step_wrong":  curr,
                    "detected_by": "sympy_factorization"
                }

    return None


# Quick sanity check
tests = [
    {"equation": "1x^2 + -6x + 0 = 0",
     "steps": ["1x^2 + -6x + 0 = 0", "x = (-(-6) ± √((-6)^2 - 4(1)(0)))/(2*1)", "36 - 0 = 40"]},
    {"equation": "1x^2 + -5x + -50 = 0",
     "steps": ["1x^2 + -5x + -50 = 0", "(x + 1)(x + 4) = 0"]},
    {"equation": "1x^2 + -6x + 0 = 0",
     "steps": ["1x^2 + -6x + 0 = 0", "(x - 6)(x - 0) = 0", "x - 6 = 0 OR x - 0 = 0", "x = 6 OR x = 0"]},
]
for t in tests:
    print(find_first_error(t))

{'equation': '1x^2 + -6x + 0 = 0', 'error_index': 2, 'step_prev': 'x = (-(-6) ± √((-6)^2 - 4(1)(0)))/(2*1)', 'step_wrong': '36 - 0 = 40', 'detected_by': 'sympy_arithmetic'}
{'equation': '1x^2 + -5x + -50 = 0', 'error_index': 1, 'step_prev': '1x^2 + -5x + -50 = 0', 'step_wrong': '(x + 1)(x + 4) = 0', 'detected_by': 'sympy_factorization'}
None


## Step 2 — Qwen3 error classifier
Called only when sympy confirms a step is wrong.

In [51]:
ERROR_TYPES = [
    "Sign error",
    "Missing root",
    "Incorrect factorization",
    "Radical simplification error",
    "Discriminant calculation error",
    "Incorrect zero product application",
    "Incorrect common factor extraction",
    "Arithmetic mistake",
    "Other"
]
error_types_str = "\n".join(f"- {e}" for e in ERROR_TYPES)

SYSTEM_PROMPT = f"""Find the error in the student's step.

Inputs:
prev: {{prev}}
wrong: {{wrong}}
op: {{operation}}
correct: {{correct}}

Error types:
{error_types_str}

Output JSON:
{{"error_type":"<type>","detail":"<short reason>","confidence":0-1}}"""


def detect_error(step_prev: str, step_wrong: str, equation: str, retries: int = 3) -> dict:
    """Use Qwen3 to identify the specific mathematical error."""
    user_msg = f"""Equation: {equation}
Previous step (correct): {step_prev}
Student's wrong step:    {step_wrong}

What is the specific mathematical error?"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0,
                response_format={"type": "json_object"}
            )
            raw = response.choices[0].message.content.strip()

            # Strip Qwen3 thinking tags
            if "</think>" in raw:
                raw = raw.split("</think>")[-1].strip()
            if "/think>" in raw:
                raw = raw.split("/think>")[-1].strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"): raw = raw[4:]
                raw = raw.strip()

            return json.loads(raw)

        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                return {"error_type": "Other", "error_detail": "Could not parse model output.", "confidence": 0.0}

## Step 3 — Full pipeline: sympy gate → Qwen3

In [53]:
def process_entry(entry: dict) -> dict | None:
    """Full pipeline for one dataset entry. Returns None if no error found."""
    error_info = find_first_error(entry)
    if error_info is None:
        return None

    llm_result = detect_error(
        step_prev=error_info["step_prev"],
        step_wrong=error_info["step_wrong"],
        equation=error_info["equation"]
    )
    return {
        "equation":     error_info["equation"],
        "step_prev":    error_info["step_prev"],
        "step_wrong":   error_info["step_wrong"],
        "detected_by":  error_info["detected_by"],
        "error_type":   llm_result.get("error_type"),
        "error_detail": llm_result.get("error_detail"),
        "confidence":   llm_result.get("confidence")
    }

## Test on individual entries

In [54]:
# Test 1: Discriminant arithmetic error
result = process_entry({
    "equation": "1x^2 + -6x + 0 = 0",
    "steps": ["1x^2 + -6x + 0 = 0",
              "x = (-(-6) ± √((-6)^2 - 4(1)(0)))/(2*1)",
              "36 - 0 = 40"]
})
print("Test 1 (expect Discriminant calculation error):")
print(json.dumps(result, indent=2))

Test 1 (expect Discriminant calculation error):
{
  "equation": "1x^2 + -6x + 0 = 0",
  "step_prev": "x = (-(-6) \u00b1 \u221a((-6)^2 - 4(1)(0)))/(2*1)",
  "step_wrong": "36 - 0 = 40",
  "detected_by": "sympy_arithmetic",
  "error_type": "Arithmetic mistake",
  "error_detail": null,
  "confidence": 0.95
}


In [55]:
# Test 2: Radical simplification error
result = process_entry({
    "equation": "1x^2 + -5x + -50 = 0",
    "steps": ["1x^2 + -5x + -50 = 0",
              "x = (-(-5) ± √((-5)^2 - 4(1)(-50)))/(2*1)",
              "√12 = 12"]
})
print("Test 2 (expect Radical simplification error):")
print(json.dumps(result, indent=2))

Test 2 (expect Radical simplification error):
{
  "equation": "1x^2 + -5x + -50 = 0",
  "step_prev": "x = (-(-5) \u00b1 \u221a((-5)^2 - 4(1)(-50)))/(2*1)",
  "step_wrong": "\u221a12 = 12",
  "detected_by": "sympy_arithmetic",
  "error_type": "Radical simplification error",
  "error_detail": null,
  "confidence": 0.95
}


In [56]:
# Test 3: Wrong factorization
result = process_entry({
    "equation": "1x^2 + -5x + -50 = 0",
    "steps": ["1x^2 + -5x + -50 = 0",
              "(x + 1)(x + 4) = 0"]
})
print("Test 3 (expect Incorrect factorization):")
print(json.dumps(result, indent=2))

Test 3 (expect Incorrect factorization):
{
  "equation": "1x^2 + -5x + -50 = 0",
  "step_prev": "1x^2 + -5x + -50 = 0",
  "step_wrong": "(x + 1)(x + 4) = 0",
  "detected_by": "sympy_factorization",
  "error_type": "Incorrect factorization",
  "error_detail": null,
  "confidence": 0.95
}


In [57]:
# Test 4: Correct entry — should return None
result = process_entry({
    "equation": "1x^2 + -6x + 0 = 0",
    "steps": ["1x^2 + -6x + 0 = 0",
              "(x - 6)(x - 0) = 0",
              "x - 6 = 0 OR x - 0 = 0",
              "x = 6 OR x = 0"]
})
print("Test 4 (expect None — no error):")
print(result)

Test 4 (expect None — no error):
None


## Run on full dataset

In [58]:
import pandas as pd

DATASET_PATH = r"C:\mariam\uni\bachelor\algebra-error-detector\notebooks\quadratic_dataset.json"

with open(DATASET_PATH) as f:
    dataset = json.load(f)

print(f"Total entries: {len(dataset)}")

# First pass: sympy only (free, instant — no API calls)
sympy_errors = [find_first_error(e) for e in dataset]
sympy_errors = [e for e in sympy_errors if e is not None]

print(f"Entries with sympy-detected errors: {len(sympy_errors)}")
print(f"Entries with no detected error:     {len(dataset) - len(sympy_errors)}")

Total entries: 2100
Entries with sympy-detected errors: 1558
Entries with no detected error:     542


In [59]:
# Second pass: Qwen3 for confirmed errors only
LIMIT = 50  # increase as needed

rows = []
for i, err_info in enumerate(sympy_errors[:LIMIT]):
    llm_result = detect_error(
        step_prev=err_info["step_prev"],
        step_wrong=err_info["step_wrong"],
        equation=err_info["equation"]
    )
    rows.append({
        "equation":     err_info["equation"],
        "step_wrong":   err_info["step_wrong"],
        "detected_by":  err_info["detected_by"],
        "error_type":   llm_result.get("error_type"),
        "error_detail": llm_result.get("error_detail"),
        "confidence":   llm_result.get("confidence")
    })

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{LIMIT} processed...")
        time.sleep(3)

df = pd.DataFrame(rows)
df

  10/50 processed...
  Attempt 1 failed: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}. Retrying in 1s...
  20/50 processed...
  30/50 processed...
  40/50 processed...
  50/50 processed...


,equation,step_wrong,detected_by,error_type,error_detail,confidence
0,1x^2 + -6x + 0 = 0,36 - 0 = 40,sympy_arithmetic,Arithmetic mistake,None,0.95
1,1x^2 + -6x + 0 = 0,√12 = 12,sympy_arithmetic,Arithmetic mistake,None,0.95
2,1x^2 + -6x + 0 = 0,(x + 1)(x + 4) = 0,sympy_factorization,Incorrect factorization,None,0.95
3,1x^2 + -5x + -50 = 0,25 - -200 = 229,sympy_arithmetic,Arithmetic mistake,None,0.95
4,1x^2 + -5x + -50 = 0,√12 = 12,sympy_arithmetic,Radical simplification error,None,0.95
5,1x^2 + -5x + -50 = 0,(x + 1)(x + 4) = 0,sympy_factorization,Incorrect factorization,None,0.95
6,3x^2 + 36x + 96 = 0,(x - -8)(x - -4) = 0,sympy_factorization,Incorrect factorization,None,0.95
7,3x^2 + 36x + 96 = 0,(x - -8)(x + 4) = 0,sympy_factorization,Incorrect common factor extraction,None,0.95
8,3x^2 + 36x + 96 = 0,1296 - 1152 = 149,sympy_arithmetic,Arithmetic mistake,None,0.95
9,3x^2 + 36x + 96 = 0,(x - -8)(x - -4) = 0,sympy_factorization,Incorrect common factor extraction,None,0.95


In [60]:
print(df["error_type"].value_counts())
print(f"\nAverage confidence: {df['confidence'].mean():.2f}")
print(f"Low confidence (<0.8): {len(df[df['confidence'] < 0.8])} rows")

error_type
Incorrect factorization               19
Arithmetic mistake                    14
Radical simplification error           9
Incorrect common factor extraction     8
Name: count, dtype: int64

Average confidence: 0.95
Low confidence (<0.8): 0 rows


## ✅ Once working → copy `process_entry()` to `src/pipeline/error_detector.py`